## Script para el modelado de temas

In [43]:
import pandas as pd
from pandas.core.interchange.from_dataframe import primitive_column_to_ndarray
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import LatentDirichletAllocation

In [44]:
df = pd.read_csv("dataset_limpio.csv")
df = df.dropna(subset=['resena_limpia'])  #quitamos nulos
textos = df['resena_limpia'].astype(str).tolist()   #convertimos los textos de cada instancia del df en una lista

#esto lo hacemos porque después de varias iteraciones nos dimos cuenta de que en todos los temas había muchas palabras como good, great, like, etcétera, que hablaban de la calidad o experiencia del usuario, y no del producto per se, entonces las vamos a quitar extendiendo las stop words
stop_words_extra = [
    'good', 'great', 'like', 'really', 'time', 'dont', 'love', 'loves',
    'best', 'use', 'buy', 'bought', 'work', 'got', 'product', 'price',
    'better', 'new', 'know', 'little', 'old', 'just', 'make'
]
stop_words = list(ENGLISH_STOP_WORDS.union(stop_words_extra))

vectorizador = CountVectorizer(max_df=0.95, min_df=2, stop_words=stop_words)  #vectorizamos y excluimos tanto palabras que están en el 95% o más de los textos y palabras que están en un solo texto (así como q casos tan aislados que deben ser un error)
matriz_tf = vectorizador.fit_transform(textos)   #vectorizamos el texto (BoW)

In [45]:
modelo_lda = LatentDirichletAllocation(  #configuramos el modelo LDA
    n_components=8,    #elegí 8 temas porque siento que puede ser variado debido a la naturaleza del dataset
    random_state=42,
    learning_method='batch'
)
modelo_lda.fit(matriz_tf)   #entrenamos

,"n_components n_components: int, default=10Number of topics... versionchanged:: 0.19 ``n_topics`` was renamed to ``n_components``",8
,"doc_topic_prior doc_topic_prior: float, default=NonePrior of document topic distribution `theta`. If the value is None,defaults to `1 / n_components`.In [1]_, this is called `alpha`.",None
,"topic_word_prior topic_word_prior: float, default=NonePrior of topic word distribution `beta`. If the value is None, defaultsto `1 / n_components`.In [1]_, this is called `eta`.",None
,"learning_method learning_method: {'batch', 'online'}, default='batch'Method used to update `_component`. Only used in :meth:`fit` method.In general, if the data size is large, the online update will be muchfaster than the batch update.Valid options:- 'batch': Batch variational Bayes method. Use all training data in each EM update. Old `components_` will be overwritten in each iteration.- 'online': Online variational Bayes method. In each EM update, use mini-batch of training data to update the ``components_`` variable incrementally. The learning rate is controlled by the ``learning_decay`` and the ``learning_offset`` parameters... versionchanged:: 0.20 The default learning method is now ``""batch""``.",'batch'
,"learning_decay learning_decay: float, default=0.7It is a parameter that control learning rate in the online learningmethod. The value should be set between (0.5, 1.0] to guaranteeasymptotic convergence. When the value is 0.0 and batch_size is``n_samples``, the update method is same as batch learning. In theliterature, this is called kappa.",0.7
,"learning_offset learning_offset: float, default=10.0A (positive) parameter that downweights early iterations in onlinelearning. It should be greater than 1.0. In the literature, this iscalled tau_0.",10.0
,"max_iter max_iter: int, default=10The maximum number of passes over the training data (aka epochs).It only impacts the behavior in the :meth:`fit` method, and not the:meth:`partial_fit` method.",10
,"batch_size batch_size: int, default=128Number of documents to use in each EM iteration. Only used in onlinelearning.",128
,"evaluate_every evaluate_every: int, default=-1How often to evaluate perplexity. Only used in `fit` method.set it to 0 or negative number to not evaluate perplexity intraining at all. Evaluating perplexity can help you check convergencein training process, but it will also increase total training time.Evaluating perplexity in every iteration might increase training timeup to two-fold.",-1
,"total_samples total_samples: int, default=1e6Total number of documents. Only used in the :meth:`partial_fit` method.",1000000.0
,"perp_tol perp_tol: float, default=1e-1Perplexity tolerance. Only used when ``evaluate_every`` is greater than 0.",0.1


In [46]:
vocab_global = vectorizador.get_feature_names_out()  #vocabulario global (corpus)
diccionario_temas = {}   #para guardar los temas

for indice_tema, tema in enumerate(modelo_lda.components_):
    #argsort() ordena de menor a mayor, nosotros re-ordeanmos en reversa
    indices_top = tema.argsort()[:-10 - 1:-1]
    palabras_clave = [vocab_global[i] for i in indices_top]  #del vocabulario global solo toma las palabras que están en el top
    nombre_tema = f"Tema {indice_tema + 1}"
    diccionario_temas[nombre_tema] = palabras_clave
    print(f"  {nombre_tema}: {', '.join(palabras_clave)}")


  Tema 1: book, read, books, information, money, waste, im, didnt, reading, pages
  Tema 2: quality, used, works, money, item, easy, amazon, purchased, doesnt, months
  Tema 3: easy, used, food, recipes, using, nice, home, book, usb, gift
  Tema 4: movie, game, film, dvd, bad, watch, movies, fun, play, money
  Tema 5: book, read, story, reading, books, life, characters, author, people, written
  Tema 6: series, book, season, video, want, way, fan, workout, episodes, episode
  Tema 7: works, son, child, year, nice, recommend, advertising, makes, excellent, using
  Tema 8: cd, album, music, songs, song, sound, band, listen, heard, voice


In [47]:
print(diccionario_temas)
temas = pd.DataFrame(diccionario_temas)
temas.to_csv("temas_lda.csv", index=False, encoding='utf-8')

{'Tema 1': ['book', 'read', 'books', 'information', 'money', 'waste', 'im', 'didnt', 'reading', 'pages'], 'Tema 2': ['quality', 'used', 'works', 'money', 'item', 'easy', 'amazon', 'purchased', 'doesnt', 'months'], 'Tema 3': ['easy', 'used', 'food', 'recipes', 'using', 'nice', 'home', 'book', 'usb', 'gift'], 'Tema 4': ['movie', 'game', 'film', 'dvd', 'bad', 'watch', 'movies', 'fun', 'play', 'money'], 'Tema 5': ['book', 'read', 'story', 'reading', 'books', 'life', 'characters', 'author', 'people', 'written'], 'Tema 6': ['series', 'book', 'season', 'video', 'want', 'way', 'fan', 'workout', 'episodes', 'episode'], 'Tema 7': ['works', 'son', 'child', 'year', 'nice', 'recommend', 'advertising', 'makes', 'excellent', 'using'], 'Tema 8': ['cd', 'album', 'music', 'songs', 'song', 'sound', 'band', 'listen', 'heard', 'voice']}


In [48]:
import pandas as pd
import numpy as np
import joblib

#definimos un diccionario de negocio
diccionario_negocio = {
    0: "Literatura (Críticas Negativas)",
    1: "Calidad y Logística (Bienes Físicos)",
    2: "Hogar, Cocina y Recetarios",
    3: "Películas y Videojuegos",
    4: "Literatura de Ficción y Narrativa",
    5: "Series y Consumo Episódico",
    6: "Productos Infantiles y Familiares",
    7: "Música y Audio"
}

probabilidades_temas = modelo_lda.transform(matriz_tf)
df['tema_id'] = np.argmax(probabilidades_temas, axis=1)

#mapear el índice numérico al nombre de negocio comprensible
df['tema_negocio'] = df['tema_id'].map(diccionario_negocio)
#guardar la confianza del modelo en esa asignación
df['probabilidad_asignacion'] = np.max(probabilidades_temas, axis=1)
#exportar el CSV final para el gráfico de distribución y filtros en Streamlit
df.to_csv('dataset_streamlit.csv', index=False, encoding='utf-8')
print("Dataset tabular exportado exitosamente para Streamlit.")

joblib.dump(modelo_lda, 'modelo_lda_prod.pkl')
joblib.dump(vectorizador, 'vectorizador_prod.pkl')
print("Artefactos binarios (.pkl) exportados exitosamente.")

Dataset tabular exportado exitosamente para Streamlit.
Artefactos binarios (.pkl) exportados exitosamente.
